# UDF vs pandas UDF: Serialization Cost

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

The same per-row transformation is implemented twice against the identical, identically-partitioned dataset: once as a row-at-a-time `udf()` and once as a vectorized `pandas_udf()` (the modern `functions.pandas_udf` scalar form). For each run: **form a hypothesis before running** (which one will be faster, and roughly how much?), run it, read `.explain(mode="formatted")` yourself, *then* call `playbook.checkpoint(df, topic="udf-pandas-udf")` and click **Reveal self-check** on the topic page to compare against the manifest-driven annotation.

Both runs' timing is measured, never hardcoded -- sourced from stage `executorRunTime` via the live `/api/v1/applications/<id>/stages` REST surface, the same "report both real numbers each run, don't assert a fixed multiplier" discipline `content/executor-tuning/` (issue #37) and `content/fault-tolerance-lineage/` (US-C9) already established.

In [ ]:
import json
import time
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("udf-pandas-udf").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def stages_snapshot():
    """Current `/api/v1/applications/<id>/stages` list -- the ground truth
    for how much wall-clock work a run's stages actually cost, the same REST
    surface the Spark UI's Stages/SQL tabs read, and the exact field
    (`executorRunTime`) this topic's manifest.yaml spotlights via
    stage_metrics. Same disposition as executor-tuning's/serialization-formats'
    own snapshot helpers."""
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def run_time_for(action_fn):
    """Runs `action_fn()`, then sums `executorRunTime` (and `numTasks`)
    across every stage that appeared as a result of it -- isolates this run's
    own cost from any prior run's stages, plus a plain wall-clock measurement
    for the same call. Both numbers are reported below, never hardcoded."""
    before_ids = {s["stageId"] for s in stages_snapshot()}
    t0 = time.time()
    action_fn()
    wall_s = time.time() - t0
    new_stages = [s for s in stages_snapshot() if s["stageId"] not in before_ids]
    run_time_ms = sum(s.get("executorRunTime", 0) for s in new_stages)
    num_tasks = sum(s.get("numTasks", 0) for s in new_stages)
    return wall_s, run_time_ms, num_tasks


SCRATCH_DIR = "/workspace/scratch/udf-pandas-udf"

## 1. Build the dataset once, shared by both runs

20,000,000 rows across 48 partitions -- large enough that per-row Python invocation overhead genuinely dominates the job (not a toy few-row example), small enough to run in a reasonable time on this project's default 3-worker/2-core/4GB-per-worker dev cluster. Written to Parquet once and re-read fresh for each run below, so both implementations see the identical on-disk data with the identical partition count -- an apples-to-apples comparison, not just "similar-sized".

In [ ]:
NUM_ROWS = 20_000_000
NUM_PARTITIONS = 48

base_df = spark.range(0, NUM_ROWS, 1, NUM_PARTITIONS).withColumnRenamed("id", "x")
base_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/base")
print(f"Wrote {NUM_ROWS} rows across {NUM_PARTITIONS} partitions to {SCRATCH_DIR}/base")

## 2. The same transformation, implemented twice

A non-trivial per-row float computation (`sin(x) * cos(x) + sqrt(|x| + 1)`) -- enough real CPU work that the comparison isn't dominated by pure per-call overhead alone, while still small enough per-row that a vectorized implementation can batch it cheaply. `row_calc` is a standard `udf()`: Spark's Python worker calls it once per row, each call crossing the JVM/Python boundary through pickle-serialized rows (the row-at-a-time path -- physically evaluated by a `BatchEvalPython` plan node, despite the name; see `concept.md` for why "batch" there refers to inter-process I/O batching, not vectorized execution). `pandas_calc` is the modern `functions.pandas_udf` scalar form: Spark hands the Python worker a whole Arrow-batched `pandas.Series` per call, and the function runs the identical math as a single vectorized NumPy expression over the whole batch -- physically evaluated by an `ArrowEvalPython` plan node.

In [ ]:
def _row_calc(x):
    import math
    return math.sin(x) * math.cos(x) + math.sqrt(abs(x) + 1.0)


row_calc = F.udf(_row_calc, DoubleType())


@F.pandas_udf(DoubleType())
def pandas_calc(s):
    import numpy as np
    return np.sin(s) * np.cos(s) + np.sqrt(np.abs(s) + 1.0)

## 3. Run 1 -- row-at-a-time `udf()`

**Hypothesis:** roughly how much slower do you expect this to be than the vectorized pandas UDF run below -- 10%? 2x? 10x? Write your answer, then run the cell.

In [ ]:
udf_df = spark.read.parquet(f"{SCRATCH_DIR}/base").withColumn("y", row_calc(F.col("x")))

udf_wall_s, udf_run_time_ms, udf_num_tasks = run_time_for(lambda: udf_df.agg(F.sum("y")).collect())
print(f"[row-udf] wall={udf_wall_s:.2f}s executorRunTime={udf_run_time_ms}ms tasks={udf_num_tasks}")
# Measured on this dev cluster: wall ~6.9s, executorRunTime ~36,100ms.

udf_df.select("y").explain(mode="formatted")

In [ ]:
checkpoint(udf_df.select("y"), topic="udf-pandas-udf")
print("Checkpoint written for the row-at-a-time udf() run -- click \'Reveal self-check\' on the topic page now, then continue below for the pandas_udf() run.")

## 4. Run 2 -- vectorized `pandas_udf()`

Same computation, same dataset, same partition count -- only the implementation changes. **Hypothesis:** confirm your answer above, then run the cell.

In [ ]:
pandas_df = spark.read.parquet(f"{SCRATCH_DIR}/base").withColumn("y", pandas_calc(F.col("x")))

pandas_wall_s, pandas_run_time_ms, pandas_num_tasks = run_time_for(lambda: pandas_df.agg(F.sum("y")).collect())
print(f"[pandas-udf] wall={pandas_wall_s:.2f}s executorRunTime={pandas_run_time_ms}ms tasks={pandas_num_tasks}")
# Measured on this dev cluster: wall ~2.2s, executorRunTime ~11,700ms.

pandas_df.select("y").explain(mode="formatted")

In [ ]:
checkpoint(pandas_df.select("y"), topic="udf-pandas-udf")
print("Checkpoint written for the pandas_udf() run -- click \'Reveal self-check\' again to compare both runs\' stages side by side.")

## 5. Compare both runs side by side

Both real numbers are printed below -- this notebook reports the measured gap, it does not assert a fixed multiplier as a hard pass/fail bar (same discipline as Executor Tuning's GC-fraction claim and Fault Tolerance & Lineage's illustrative retry count). The only hard assertions are the directional claim (pandas UDF measurably faster) and the plan-shape claim (each implementation prints its own distinct operator) -- not an exact ratio, since that number will vary run to run and machine to machine.

In [ ]:
header = " " * 16 + "wall (s)".rjust(10) + "executorRunTime (ms)".rjust(22) + "tasks".rjust(8)
print(header)
print(f"{'row-udf':16s}{udf_wall_s:>10.2f}{udf_run_time_ms:>22d}{udf_num_tasks:>8d}")
print(f"{'pandas-udf':16s}{pandas_wall_s:>10.2f}{pandas_run_time_ms:>22d}{pandas_num_tasks:>8d}")

wall_speedup = udf_wall_s / pandas_wall_s if pandas_wall_s else float("inf")
run_time_speedup = udf_run_time_ms / pandas_run_time_ms if pandas_run_time_ms else float("inf")
print(f"\nMeasured speedup this run -- wall-clock: {wall_speedup:.2f}x, executorRunTime: {run_time_speedup:.2f}x")
# Measured on this dev cluster across independent runs: ~3.0-3.2x on both wall-clock and
# executorRunTime -- consistent direction and rough magnitude, not asserted as exact.

assert pandas_wall_s < udf_wall_s, "expected the vectorized pandas UDF run to be measurably faster (wall-clock)"
assert pandas_run_time_ms < udf_run_time_ms, "expected the vectorized pandas UDF run to be measurably faster (executorRunTime)"

import contextlib
import io

buf_udf = io.StringIO()
with contextlib.redirect_stdout(buf_udf):
    udf_df.select("y").explain(mode="formatted")
buf_pandas = io.StringIO()
with contextlib.redirect_stdout(buf_pandas):
    pandas_df.select("y").explain(mode="formatted")

print("\nrow-udf plan node:   BatchEvalPython present =", "BatchEvalPython" in buf_udf.getvalue())
print("pandas-udf plan node: ArrowEvalPython present =", "ArrowEvalPython" in buf_pandas.getvalue())
assert "BatchEvalPython" in buf_udf.getvalue(), "expected the row-at-a-time udf() plan to show a BatchEvalPython node"
assert "ArrowEvalPython" in buf_pandas.getvalue(), "expected the pandas_udf() plan to show an ArrowEvalPython node"
assert "ArrowEvalPython" not in buf_udf.getvalue(), "row-udf plan should not show ArrowEvalPython"
assert "BatchEvalPython" not in buf_pandas.getvalue(), "pandas-udf plan should not show BatchEvalPython"